# Import Data

In [10]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random
import warnings
import torch
import multiprocessing
import os


class CFG:
    
    target_name='tag'
    seed = 58
    
    #######################################################################################
    # GPU
    gpu_available = torch.cuda.is_available()

    print(f"CUDA available: {gpu_available}")
    if gpu_available:
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
        print(f"Number of GPUs: {torch.cuda.device_count()}")
    else:
        print(f'Use CPU, \nNumber of CPUs: {multiprocessing.cpu_count()}')
    
    #######################################################################################
    # Seed
    
    @staticmethod
    def seed_all(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


CFG.seed_all(CFG.seed)

warnings.simplefilter('ignore')
sns.set_theme(style="ticks")

CUDA available: False
Use CPU, 
Number of CPUs: 4


In [11]:
x = pd.read_csv('/kaggle/input/datasets/artsmirnovch/spd-feature-set-2/x_train.csv')
y = pd.read_csv('/kaggle/input/datasets/artsmirnovch/spd-feature-set-2/y_train.csv')
x = x.drop('Unnamed: 0', axis=1)
y = y.drop('Unnamed: 0', axis=1)

# Prepare data

In [12]:
from sklearn.model_selection import train_test_split


x_train, x_val, y_train, y_val = train_test_split(
    x, y, stratify=y, shuffle=True, test_size=0.2, random_state=CFG.seed
)

# Optuna

In [13]:
import optuna

import ydf

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def objective(trial):

    n_jobs_setting = 4

    param_space = {
        'num_trees': trial.suggest_int('num_trees', 100, 3000, step=50),
        'max_depth': trial.suggest_int('depth', 3, 14),
        'min_examples': trial.suggest_int('min_examples', 5, 30),
        'num_candidate_attributes_ratio': trial.suggest_float('num_candidate_attrs_ratio', 0.5, 0.9),
        'early_stopping_num_trees_look_ahead': trial.suggest_int('early_stopping_num_trees_look_ahead', 10, 200, step=10),
        'l1_regularization': trial.suggest_float('l1_regularization', 1e-5, 2, log=True),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-5, 2, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1),
        'shrinkage': trial.suggest_float('shrinkage', 1e-4, 1e-1, log=True),
    }

    # Var 1
    # local_x_train, local_x_val, local_y_train, local_y_val = train_test_split(
    #     x, y, stratify=y, shuffle=True, test_size=0.25
    # )
    # clf = XGBClassifier(**params).fit(local_x_train, local_y_train, eval_set=[(local_x_val, local_y_val)], verbose=0)
    # y_pred_proba = clf.predict_proba(local_x_val)[:, 1]
    # auc_score = roc_auc_score(local_y_val, y_pred_proba)
    # return {'score': auc_score, 'status': STATUS_OK}
    
    # Var 2
    seed = CFG.seed # solve ModuleNotFoundError
    
    train_df = x_train.copy()
    train_df['target'] = y_train.copy()

    learner = ydf.GradientBoostedTreesLearner(
        label="target",
        task=ydf.Task.CLASSIFICATION,
        early_stopping='LOSS_INCREASE',
        validation_ratio=0.2,
        random_seed=CFG.seed,
        num_threads=n_jobs_setting,
        **param_space
    )

    model = learner.train(train_df)
        
    train_pred_proba = model.predict(x_train)
    val_pred_proba = model.predict(x_val)
        
    train_score = roc_auc_score(y_train, train_pred_proba)
    val_score = roc_auc_score(y_val, val_pred_proba)

    trial.set_user_attr("train_scores_mean", train_score)

    return val_score


study = optuna.create_study(
    direction='maximize', 
    study_name='ydf_gbt_optimization',
    load_if_exists=True
)

study.optimize(objective, n_trials=600, timeout=3*3600, n_jobs=-1, show_progress_bar=True)

[I 2026-03-01 14:19:56,079] A new study created in memory with name: ydf_gbt_optimization


  0%|          | 0/600 [00:00<?, ?it/s]

Train model on 78409 examples
Train model on 78409 examples
Train model on 78409 examples
Train model on 78409 examples
Model trained in 0:01:30.840039
[I 2026-03-01 14:21:28,121] Trial 1 finished with value: 0.9190013943795043 and parameters: {'num_trees': 150, 'depth': 6, 'min_examples': 29, 'num_candidate_attrs_ratio': 0.8448379245608393, 'early_stopping_num_trees_look_ahead': 110, 'l1_regularization': 1.27615693072494e-05, 'l2_regularization': 0.007881689322564802, 'subsample': 0.9468626988565014, 'shrinkage': 0.0013606240980900519}. Best is trial 1 with value: 0.9190013943795043.
Train model on 78409 examples
Model trained in 0:02:11.377238
[I 2026-03-01 14:22:08,977] Trial 2 finished with value: 0.9241852159943575 and parameters: {'num_trees': 250, 'depth': 6, 'min_examples': 17, 'num_candidate_attrs_ratio': 0.6577447295969993, 'early_stopping_num_trees_look_ahead': 180, 'l1_regularization': 1.0143365023014139e-05, 'l2_regularization': 0.003776292656009485, 'subsample': 0.9991465

In [15]:
best_trial = study.best_trial
print(f"Best validation score: {best_trial.value}")
print(f"Best train score: {best_trial.user_attrs['train_scores_mean']}")

best_params = study.best_params
print("Best parameters:", best_params)

Best validation score: 0.9567912787083855
Best train score: 0.9853513820355633
Best parameters: {'num_trees': 2250, 'depth': 10, 'min_examples': 11, 'num_candidate_attrs_ratio': 0.8022988236732308, 'early_stopping_num_trees_look_ahead': 80, 'l1_regularization': 5.741154575969731e-05, 'l2_regularization': 0.00012891946724916293, 'subsample': 0.6794172580446142, 'shrinkage': 0.004848652484384624}


In [17]:
import plotly.graph_objects as go


trials_df = study.trials_dataframe()
val_scores = trials_df['value'].values
train_scores = [t.user_attrs['train_scores_mean'] for t in study.trials]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(len(val_scores))),
    y=val_scores,
    mode='markers+lines',
    name='Validation Score',
    marker={'color': 'blue'}
))

fig.add_trace(go.Scatter(
    x=list(range(len(train_scores))),
    y=train_scores,
    mode='markers+lines',
    name='Train Score',
    marker={'color': 'red'}
))

fig.update_layout(
    title='Optimization History - Train vs Validation Scores',
    xaxis_title='Trial',
    yaxis_title='AUC Score',
    hovermode='x unified'
)

fig.write_html("optimization_history.html")
fig.show()